# debug_inference
起動・デバッグ用のノートブック。処理の本体は `app/`（アプリ）と `scripts/`（成果物の生成）にあり、ここでは呼び出すだけ。

コマンド1つで起動したい場合は `python run.py`（`--tunnel` で外部公開）でも同じことができる。

## 準備（初回のみ）

In [ ]:
# ## googlecolab なら実行
# from google.colab import drive
# drive.mount("/content/drive/")
# %cd "/content/drive/MyDrive/Colab Notebooks/app_labGuide/backend"

In [ ]:
# ## googlecolab なら実行（研究室サーバの場合は初回のみでOK）
# # 1. システムのセットアップ
# !bash ../setup.sh

# # 2. Pythonライブラリのインストール
# !pip install -r ../requirements.txt

# # 3. voicevox用に権限付与（権限エラーが出たときの対策）
# !chmod +x /content/drive/MyDrive/Colab\ Notebooks/app_labGuide/tools/linux-cpu-x64/run

In [ ]:
# 4. ベクトルデータベース作成するときはコメント解除してください（.txtドキュメントが無い場合の実行した場合エラーが発生しますが、その後の操作に影響はありません）
# !python3 app/rag/build_index.py

## 設定とモデルのロード
設定値（モデル・層・成果物の場所）は `run.py` の先頭にまとめてある。変える場合はそちらを編集する。

In [ ]:
import run   # backend/ に移動し、sys.path を設定する
from run import (MODEL_ID, RESEARCH_MODEL_KEY, TARGET_LAYER, ASSETS_ROOT,
                 build_manager, start_voicevox, start_server)
from app.core.llm_engine import LLMEngine

print("model :", MODEL_ID)
print("layer :", TARGET_LAYER, "(hidden_states のインデックス)")
print("assets:", ASSETS_ROOT)

In [ ]:
# 量子化しない（活性化の値が変わり、成果物の avg_norm・α上限と対応しなくなる）
engine = LLMEngine(model_id=MODEL_ID, quantization_8bit=False)

## 研究成果物の作り直し（任意・通常は不要）
同梱の `data/processed/steering_assets/` をそのまま使うなら実行しなくてよい。

| ステップ | 内容 | 必要なもの |
|---|---|---|
| activations | 合成データの隠れ状態を抽出してキャッシュ | GPU・合成データ |
| vectors | CAA＋中立PCAで感情ステアリングベクトル | activations のキャッシュ |
| probe | 線形感情プローブ（8感情＋neutral） | activations のキャッシュ |
| refusal | 拒絶方向 r_hat | GPU・拒絶ペアCSV |

ロード済みの `engine` を使い回すので、モデルを二重に読み込まない。α上限はここでは作らない（研究リポジトリで決めた値を使う）。

In [ ]:
REBUILD_ASSETS = False
if REBUILD_ASSETS:
    from scripts.build_assets import build_all
    build_all(engine, model_key=RESEARCH_MODEL_KEY, layer=TARGET_LAYER,
              steps=["activations", "vectors", "probe"])   # "refusal" は拒絶ペアCSVがある場合のみ

## マネージャの作成

In [ ]:
manager = build_manager(engine, use_rag=True)

### 動作確認（UIを使わずに試す）

In [ ]:
# 感情プローブだけを試す（内部状態は変わらない）
for t in ["この前話してた本、やっと読み終わったよ",
          "大学の学食について教えてください",
          "この対応は本当にひどい"]:
    print(t, "->", manager.classifier.predict_top(t, k=3),
          f"neutral={manager.classifier.neutral_score(t):.2f}")

In [ ]:
# 1ターン生成する（内部状態と履歴が更新される。終わったら reset する）
res = manager.generate_reply("この前話してた本、やっと読み終わったよ", condition="steering")
print(res["reply"])
print("emotion:", res["complex_emotion"], " alpha:", res["applied_alpha"])
manager.reset_conversation()

## サーバの起動

In [ ]:
voicevox = start_voicevox()          # エンジンが無ければ None（音声なしで動く）
app = start_server(manager, voicevox, port=8000, tunnel=True, background=True, rag=True)
# tunnel=False にすると http://localhost:8000 のみで起動する